In [0]:
SELECT *
FROM workspace.bright_coffee.bright_coffee_shop_analysis;

SELECT 
    product_id,
    product_detail AS product_name,
    product_category AS category_name,
    unit_price AS price
FROM workspace.bright_coffee.bright_coffee_shop_analysis;

DESCRIBE bright_coffee_shop.default.bright_coffee_shop_analysis;


SELECT 
    product_id,
    product_detail AS product_name,
    product_category AS category_name,
    unit_price AS price
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;


SELECT transaction_qty, ROUND(SUM (CAST(transaction_qty AS DOUBLE) * CAST (REPLACE(unit_price,',','.')AS DOUBLE)),0) AS Total_Amount
FROM bright_coffee_shop.default.bright_coffee_shop_analysis
GROUP BY transaction_qty;

-- Checking the product Cat--
SELECT DISTINCT product_category
FROM bright_coffee_shop.default.bright_coffee_shop_analysis
LIMIT 100;

-- Cleaning Product Cat--
SELECT DISTINCT 
      product_category,
      CASE 
          WHEN product_category IS NULL THEN 'unknown'
          WHEN product_category =' ' THEN 'unknown'
          ELSE product_category
      END AS Product_cat
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

-- Inspecting Produc Type--
SELECT DISTINCT product_type
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

--Cleaning Product Type--
SELECT DISTINCT 
      product_type,
      CASE 
          WHEN product_type IS NULL THEN 'unknown'
          WHEN product_type =' ' THEN 'unknown'
          ELSE product_type
      END AS Product_typ
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

SELECT transaction_qty, ROUND(SUM (CAST(transaction_qty AS DOUBLE) * CAST (REPLACE(unit_price,',','.')AS DOUBLE)),0) AS Total_Amount
FROM bright_coffee_shop.default.bright_coffee_shop_analysis
GROUP BY transaction_qty;

CREATE OR REPLACE TABLE bright_coffee_sales_clean AS

SELECT
    transaction_id,

    CAST(transaction_date AS DATE) AS transaction_date,

    CAST(transaction_time AS TIMESTAMP) AS transaction_time,

    transaction_qty,

    store_id,
    store_location,

    product_id,
    product_category,
    product_type,
    product_detail,

    CAST(unit_price AS DECIMAL(10,2)) AS unit_price,

    -- Revenue
    CAST(unit_price AS DECIMAL(10,2)) * transaction_qty
        AS total_amount,

    -- Hour of transaction
    HOUR(CAST(transaction_time AS TIMESTAMP))
        AS transaction_hour,

    -- 30-minute bucket
    CONCAT(
        LPAD(
            CAST(
                FLOOR(
                    HOUR(CAST(transaction_time AS TIMESTAMP))
                    * 2
                    +
                    MINUTE(CAST(transaction_time AS TIMESTAMP)) / 30
                ) / 2
                AS INT
            ),
            2,
            '0'
        ),
        ':',
        CASE
            WHEN MINUTE(CAST(transaction_time AS TIMESTAMP)) < 30
            THEN '00'
            ELSE '30'
        END
    ) AS transaction_time_bucket

FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

--Data Quality checks
SELECT  
    COUNT(*) AS total_rows, 
    COUNT(DISTINCT transaction_id) AS unique_transactions,  
    COUNT(DISTINCT product_id) AS unique_products,  
    COUNT(DISTINCT store_id) AS unique_stores,  
    
    MIN(transaction_date) AS first_transaction_date,    
    MAX(transaction_date) AS last_transaction_date, 
        SUM(transaction_qty) AS total_units,    
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM bright_coffee_sales_clean; 

--Checking for Total Revenue
SELECT
    ROUND(SUM(total_amount), 2) AS total_revenue,
    SUM(transaction_qty) AS total_units_sold,
    COUNT(DISTINCT transaction_id) AS total_transactions,
    ROUND(
        SUM(total_amount) / COUNT(DISTINCT transaction_id),
        2
    ) AS average_transaction_value
FROM bright_coffee_sales_clean;

--Checking the product types that gives most revenue
SELECT
    product_category,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold,
    COUNT(DISTINCT transaction_id) AS transactions,
    ROUND(
        SUM(total_amount)
        /
        SUM(SUM(total_amount)) OVER ()
        * 100,
        2
    ) AS revenue_percentage
FROM bright_coffee_sales_clean
GROUP BY product_category
ORDER BY revenue DESC;

--checking which product generates most revenue
SELECT
    product_type,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold,
    COUNT(DISTINCT transaction_id) AS transactions
FROM bright_coffee_sales_clean
GROUP BY product_type
ORDER BY revenue DESC
LIMIT 10;

--checking top 10 products by units sold
SELECT
    product_type,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY product_type
ORDER BY units_sold DESC
LIMIT 10;

--checking top 10 product details
SELECT
    product_detail,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue,
    ROUND(
        SUM(total_amount) /
        SUM(transaction_qty),
        2
    ) AS average_price
FROM bright_coffee_sales_clean
GROUP BY product_detail
ORDER BY revenue DESC
LIMIT 10;

--checking for low performing products
SELECT
    product_detail,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY product_detail
HAVING SUM(transaction_qty) >= 100
ORDER BY revenue ASC
LIMIT 10;

--Checking time of the day where there is high performance
SELECT
    transaction_hour,
    COUNT(DISTINCT transaction_id) AS transactions,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY transaction_hour
ORDER BY transaction_hour;

--checking the most peak in 30min interval
SELECT
    transaction_time_bucket,
    COUNT(DISTINCT transaction_id) AS transactions,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY transaction_time_bucket
ORDER BY revenue DESC;

--checking for slow sales period
SELECT
    transaction_time_bucket,
    COUNT(DISTINCT transaction_id) AS transactions,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY transaction_time_bucket
ORDER BY revenue ASC
LIMIT 10;

--checking the store that makes most revenue
SELECT
    store_id,
    store_location,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold,
    COUNT(DISTINCT transaction_id) AS transactions,
    ROUND(
        SUM(total_amount) /
        COUNT(DISTINCT transaction_id),
        2
    ) AS average_transaction_value
FROM bright_coffee_sales_clean
GROUP BY store_id, store_location
ORDER BY revenue DESC;

--checking the growth of the business
SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS month,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold,
    COUNT(DISTINCT transaction_id) AS transactions
FROM bright_coffee_sales_clean
GROUP BY DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY month;

--checking days of the week analysis
SELECT
    DAYOFWEEK(transaction_date) AS day_number,
    DATE_FORMAT(transaction_date, 'EEEE') AS day_name,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold
FROM bright_coffee_sales_clean
GROUP BY
    DAYOFWEEK(transaction_date),
    DATE_FORMAT(transaction_date, 'EEEE')
ORDER BY day_number;

--checking revenue per month
SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS month,
    product_category,
    ROUND(SUM(total_amount),2) AS revenue
FROM bright_coffee_sales_clean
GROUP BY
    DATE_FORMAT(transaction_date, 'yyyy-MM'),
    product_category
ORDER BY month, revenue DESC;

-- checking for product at time analysis
SELECT
    transaction_hour,
    product_category,
    ROUND(SUM(total_amount),2) AS revenue,
    SUM(transaction_qty) AS units_sold
FROM bright_coffee_sales_clean
GROUP BY
    transaction_hour,
    product_category
    
ORDER BY
    transaction_hour,
    revenue DESC;

    --Checking product against revenue
    SELECT
    product_type,
    SUM(transaction_qty) AS units_sold,
    ROUND(SUM(total_amount),2) AS revenue,
    ROUND(
        SUM(total_amount) / SUM(transaction_qty),
        2
    ) AS revenue_per_unit
FROM bright_coffee_sales_clean
GROUP BY product_type
ORDER BY revenue DESC;

--Creating the Revenue column
SELECT transaction_qty,
        product_type,
        product_category,
        unit_price,
        (unit_price*transaction_qty) AS Total_Amount
FROM bright_coffee.analytics.coffee_sales;

SELECT transaction_qty,
        product_type,
        product_category,
        unit_price,
        ROUND(SUM(unit_price*transaction_qty),2) AS Total_Amount
FROM bright_coffee.analytics.coffee_sales
GROUP BY transaction_qty, unit_price,product_type,
        product_category;


SELECT SUM(DISTINCT product_category),transaction_qty,
        product_type, unit_price
FROM bright_coffee.analytics.coffee_sales
GROUP BY transaction_qty, unit_price,product_type,
        product_category;


--
SELECT product_category, ROUND(SUM(unit_price*transaction_qty),2) AS Total_revenue
FROM bright_coffee.analytics.coffee_sales
GROUP BY product_category
ORDER BY Total_revenue DESC;


--CODE TO CONVERT DATATYPE TO ANOTHER (string to numeric)
SUM(transaction_qty) AS total_daily_sales,
       ROUND(SUM(CAST(transaction_qty AS DOUBLE) * CAST (REPLACE (unit_price, ',', '.') AS DOUBLE)), 0) AS  total_amount
FROM bright_coffee.analytics.coffee_sales
GROUP BY transaction_qty;
---------------------------------------------------------------------------------------------------
--DATA CLEANING
---------------------------------------------------------------------------------------------

--Checking duplicates
SELECT transaction_id,
        COUNT(*) AS Duplicate_cnt
FROM bright_coffee_shop.default.bright_coffee_shop_analysis
GROUP BY transaction_id
HAVING COUNT(*)>1;

--FINAL CODE FOR CHECKING DUPLICATES FOR ALL COLUMNS
SELECT *,
        COUNT(*) AS Duplicate_cnt
FROM bright_coffee_shop.default.bright_coffee_shop_analysis
GROUP BY ALL
HAVING COUNT(*)>1;

--CHECKING THE DATE COLUMN

SELECT DISTINCT transaction_date
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

SELECT DISTINCT DATE_FORMAT(transaction_date, 'MMMM') AS Month_name
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

--CHECKING TRANCTION TIME COLUMN
SELECT DISTINCT transaction_time
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

SELECT DISTINCT DATE_FORMAT(transaction_time, 'HH:MM:SS') AS Time
FROM bright_coffee_shop.default.bright_coffee_shop_analysis;

--Creating time buckets
SELECT DISTINCT 
    WHEN Time (transaction_time) BETWEEN 07:00:00 AND 11:59:00 THEN Morinng


:param_1    CREATE OR REPLACE VIEW bright_coffee_dashboard AS
SELECT
    *,
        -- DATE
    CAST(transaction_date AS DATE) AS date,
    -- MONTH NUMBER
    MONTH(transaction_date) AS month_number,
    -- MONTH NAME
    DATE_FORMAT(transaction_date, 'MMMM') AS month_name,
    -- YEAR-MONTH
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS year_month,
    -- DAY
    DATE_FORMAT(transaction_date, 'EEEE') AS day_name,
    -- HOUR
    HOUR(transaction_time) AS hour,
    -- 30-MINUTE INTERVAL
    CONCAT(
        DATE_FORMAT(transaction_time, 'HH'),
        ':',
        CASE
            WHEN MINUTE(transaction_time) < 30 THEN '00'
            ELSE '30'
        END
    ) AS thirty_minute_interval,
    -- CATEGORY
    CASE
        WHEN LOWER(product_type) LIKE '%coffee%'
          OR LOWER(product_type) LIKE '%espresso%'
        THEN 'Coffee'
        WHEN LOWER(product_type) LIKE '%tea%'
          OR LOWER(product_type) LIKE '%chai%'
        THEN 'Tea'
        WHEN LOWER(product_type) LIKE '%chocolate%'
        THEN 'Chocolate'
        WHEN LOWER(product_type) LIKE '%scone%'
          OR LOWER(product_type) LIKE '%sandwich%'
          OR LOWER(product_type) LIKE '%pastry%'
          OR LOWER(product_type) LIKE '%cake%'
          OR LOWER(product_type) LIKE '%cookie%'
          OR LOWER(product_type) LIKE '%muffin%'
        THEN 'Food'
        ELSE 'Other'
    END AS category
FROM bright_coffee_sales_clean;

SELECT
    HOUR(transaction_time) AS hour,
    CASE
        WHEN MINUTE(transaction_time) < 30 THEN 0
        ELSE 30
    END AS minute_group,
    CONCAT(
        DATE_FORMAT(transaction_time, 'HH'),
        ':',
        CASE
            WHEN MINUTE(transaction_time) < 30 THEN '00'
            ELSE '30'
        END
    ) AS thirty_minute_interval,
    SUM(total_amount) AS total_revenue,
    SUM(transaction_qty) AS total_units,
    COUNT(DISTINCT transaction_id) AS transactions
FROM bright_coffee_sales_clean
GROUP BY
    HOUR(transaction_time),
    CASE
        WHEN MINUTE(transaction_time) < 30 THEN 0
        ELSE 30
    END,
    CONCAT(
        DATE_FORMAT(transaction_time, 'HH'),
        ':',
        CASE
            WHEN MINUTE(transaction_time) < 30 THEN '00'
            ELSE '30'
        END
    )

ORDER BY
    hour,
    minute_group;


    CREATE OR REPLACE VIEW bright_coffee_ceo_dashboard AS

SELECT

    /* =========================
       TRANSACTION DIMENSIONS
       ========================= */

    transaction_id,

    CAST(transaction_date AS DATE) AS date,

    DATE_FORMAT(transaction_date,'yyyy-MM') AS month,

    DATE_FORMAT(transaction_date,'MMMM') AS month_name,

    DATE_FORMAT(transaction_date,'EEEE') AS day_name,

    store_location AS store,

    product_type,

    transaction_qty AS units_sold,

    total_amount AS revenue,


    /* =========================
       TIME DIMENSIONS
       ========================= */

    HOUR(transaction_time) AS hour,

    MINUTE(transaction_time) AS minute,


    /* 30-MINUTE INTERVAL */

    CONCAT(
        DATE_FORMAT(transaction_time,'HH'),
        ':',
        CASE
            WHEN MINUTE(transaction_time) < 30
            THEN '00'
            ELSE '30'
        END
    ) AS thirty_minute_interval,


    /* =========================
       PRODUCT CATEGORY
       ========================= */

    CASE

        WHEN LOWER(product_type) LIKE '%coffee%'
          OR LOWER(product_type) LIKE '%espresso%'
        THEN 'Coffee'

        WHEN LOWER(product_type) LIKE '%tea%'
          OR LOWER(product_type) LIKE '%chai%'
        THEN 'Tea'

        WHEN LOWER(product_type) LIKE '%chocolate%'
        THEN 'Chocolate'

        WHEN LOWER(product_type) LIKE '%scone%'
          OR LOWER(product_type) LIKE '%sandwich%'
          OR LOWER(product_type) LIKE '%pastry%'
          OR LOWER(product_type) LIKE '%cake%'
          OR LOWER(product_type) LIKE '%cookie%'
          OR LOWER(product_type) LIKE '%muffin%'
        THEN 'Food'

        ELSE 'Other'

    END AS category,


    /* =========================
       COMBINATION DIMENSIONS
       ========================= */

    CONCAT(
        store_location,
        ' - ',
        LPAD(HOUR(transaction_time),2,'0'),
        ':00'
    ) AS store_hour,


    CONCAT(
        CASE
            WHEN LOWER(product_type) LIKE '%coffee%'
              OR LOWER(product_type) LIKE '%espresso%'
            THEN 'Coffee'

            WHEN LOWER(product_type) LIKE '%tea%'
              OR LOWER(product_type) LIKE '%chai%'
            THEN 'Tea'

            WHEN LOWER(product_type) LIKE '%chocolate%'
            THEN 'Chocolate'

            WHEN LOWER(product_type) LIKE '%scone%'
              OR LOWER(product_type) LIKE '%sandwich%'
              OR LOWER(product_type) LIKE '%pastry%'
              OR LOWER(product_type) LIKE '%cake%'
              OR LOWER(product_type) LIKE '%cookie%'
              OR LOWER(product_type) LIKE '%muffin%'
            THEN 'Food'
            ELSE 'Other'
        END,

        ' - ',

        LPAD(HOUR(transaction_time),2,'0'),
        ':00'

    ) AS category_hour
FROM bright_coffee_sales_clean;

SELECT *
FROM bright_coffee_ceo_dashboard
LIMIT 100;

SELECT *
FROM bright_coffee_ceo_dashboard;